# Project Overview

## Objective
The objective of this project is to price a credit-risky fixed-income instrument using a reduced-form (intensity-based) credit risk model. The project focuses on translating observed market credit spreads into implied default intensities, valuing survival-weighted cash flows, and quantifying key credit risk sensitivities.

---

## Methodology

The project follows a reduced-form credit pricing framework with constant hazard rates:

### 1. Instrument Specification
- Fixed notional bond with maturity \( T \)
- Periodic coupon payments
- Flat risk-free discount rate
- Constant recovery rate assumption

### 2. Hazard Rate Calibration
Default intensity is inferred from the observed credit spread using:
$$
\lambda = \frac{\text{Credit Spread}}{1 - R}
$$
where \( R \) denotes the recovery rate.

### 3. Survival and Default Probabilities
- Survival probabilities are modeled using an exponential survival function:
$$
S(t) = e^{-\lambda t}
$$
- Interval default probabilities are computed from successive survival probabilities over coupon periods.

### 4. Cash Flow Valuation
Expected present value is decomposed into:
- Survival-weighted coupon payments
- Survival-weighted principal repayment
- Expected recovery payments in default scenarios

All cash flows are discounted using the risk-free rate.

### 5. Fair Coupon Calibration
The fair (par) coupon rate is solved numerically such that the bond price equals par value, using a root-finding procedure.

### 6. Credit Risk Sensitivities
- Credit spread sensitivity (CS01) is computed via small spread shifts.
- Jump-to-Default (JTD) is calculated as loss given default multiplied by notional.

---

## Model Outputs and Interpretation Framework

The outputs produced by this notebook represent the bond valuation and credit risk characteristics implied by the chosen model inputs.

They should be interpreted as follows:
- The bond price reflects expected discounted cash flows adjusted for default risk.
- The fair coupon represents the compensation required to price the instrument at par under the assumed default intensity.
- CS01 measures sensitivity to marginal changes in credit spreads.
- Jump-to-Default captures the immediate loss associated with a default event.

Changes in credit spreads, recovery assumptions, or discount rates will directly affect these outputs.

In [24]:
import numpy as np
from scipy.optimize import brentq

# ---- Deal terms ----
N = 1000000            # notional
T = 5.0                # maturity in years
freq = 4               # quarterly coupons
dt = 1/freq
R = 0.40               # recovery
LGD = 1 - R

# ---- Market inputs (simple first) ----
r = 0.04               # flat risk-free rate (4%)
spread = 0.02          # credit spread proxy (200 bps)

print("Notional:", N, "Maturity:", T, "freq:", freq, "r:", r, "spread:", spread, "Recovery:", R)

Notional: 1000000 Maturity: 5.0 freq: 4 r: 0.04 spread: 0.02 Recovery: 0.4


In [14]:
times = np.arange(dt, T + 1e-9, dt)   # 0.25, 0.50, ..., 5.00
n_payments = len(times)

times[:5], times[-3:], n_payments

(array([0.25, 0.5 , 0.75, 1.  , 1.25]), array([4.5 , 4.75, 5.  ]), 20)

In [15]:
DF = np.exp(-r * times)
DF[:5]

array([0.99004983, 0.98019867, 0.97044553, 0.96078944, 0.95122942])

In [16]:
h = spread / LGD
h

0.03333333333333333

In [17]:
S = np.exp(-h * times)
S[:5]

array([0.99170129, 0.98347145, 0.97530991, 0.9672161 , 0.95918946])

In [18]:
S_prev = np.concatenate([[1.0], S[:-1]])  # S(0)=1
p_default_interval = S_prev - S           # default in (t_{k-1}, t_k]
p_default_interval[:5], p_default_interval.sum()

(array([0.00829871, 0.00822984, 0.00816154, 0.00809381, 0.00802664]),
 0.15351827510938587)

In [19]:
def price_cln(coupon_rate):
    coupon_cash = N * coupon_rate * dt
    
    # expected coupons + principal (only if survive to payment date)
    pv_coupons = np.sum(DF * S * coupon_cash)
    pv_principal = DF[-1] * S[-1] * N
    
    # default leg: discount at interval midpoints
    t_mid = times - dt/2
    DF_mid = np.exp(-r * t_mid)
    pv_default_loss = np.sum(DF_mid * p_default_interval * LGD * N)
    
    price = pv_coupons + pv_principal - pv_default_loss
    return price, pv_coupons, pv_principal, pv_default_loss

# Example: try coupon = 6%
price, pv_cpn, pv_prin, pv_def = price_cln(0.06)
price, pv_cpn, pv_prin, pv_def

(858178.7783988608, 248853.42266267093, 693040.6200864415, 83715.26435025166)

In [20]:
def fair_coupon():
    f = lambda c: price_cln(c)[0] - N
    return brentq(f, 0.0001, 0.50)  # search between 1bp and 50%

c_star = fair_coupon()
c_star

0.09419391706580206

In [21]:
price_star, pv_cpn_star, pv_prin_star, pv_def_star = price_cln(c_star)

print("FAIR COUPON (annual):", round(c_star*100, 4), "%")
print("Price at fair coupon:", round(price_star, 2))
print("PV coupons:", round(pv_cpn_star, 2))
print("PV principal:", round(pv_prin_star, 2))
print("PV default loss:", round(pv_def_star, 2))

FAIR COUPON (annual): 9.4194 %
Price at fair coupon: 1000000.0
PV coupons: 390674.64
PV principal: 693040.62
PV default loss: 83715.26


In [22]:
def price_with_rate_and_spread(coupon_rate, r_new, spread_new):
    LGD = 1 - R
    h_new = spread_new / LGD
    S_new = np.exp(-h_new * times)
    S_prev_new = np.concatenate([[1.0], S_new[:-1]])
    p_def_new = S_prev_new - S_new
    
    DF_new = np.exp(-r_new * times)
    coupon_cash = N * coupon_rate * dt
    
    pv_cpn = np.sum(DF_new * S_new * coupon_cash)
    pv_prin = DF_new[-1] * S_new[-1] * N
    
    t_mid = times - dt/2
    DF_mid = np.exp(-r_new * t_mid)
    pv_def = np.sum(DF_mid * p_def_new * (1-R) * N)
    
    return pv_cpn + pv_prin - pv_def

# Base price at fair coupon
P0 = price_with_rate_and_spread(c_star, r, spread)

# DV01 approximation
bp = 0.0001
P_up = price_with_rate_and_spread(c_star, r + bp, spread)
DV01 = P_up - P0   # price change for +1bp

DV01

-423.3767093701754

In [11]:
P_spread_up = price_with_rate_and_spread(c_star, r, spread + bp)
CS01 = P_spread_up - P0
CS01

-1123.9324178870302

In [23]:
JTD_loss = LGD * N
JTD_loss

600000.0